# clock/04.4 — OOD generalization of the completion signal

Load the attention heads selected by a completion-mask run trained on the **Python**
condition-check dataset, compute a **per-head mean** intervention signal (§6.1 of `04.3_eval`),
then check whether the same signal transfers off-the-shelf to:

1. **In-distribution sanity** — python `build_example` (should look the same as `04.3`).
2. **Same shape, new language** — cpp `build_example` (new `lang="cpp"` support in `condition_data.py` + `multiline_prompts.py`).
3. **Hardcoded C++ prompts** — small, hand-written C++ programs, more freeform than `build_example`'s `check_conditions` shape.

The mask is **NOT retrained** on any of these — that's the whole point. This is an off-the-shelf
generalization test: does the intervention direction learned on python condition-check still
push a dest toward the flipped verdict when the dest is C++, or a different shape entirely?

Follows the `04.3_eval` conventions:
- augmentation `code = code.replace("\n", "    \n") + "    "`
- `PIVOT_POS = -2` (the `"    \n"` pivot)
- readout at position `-1` (never at the patched `-2`)


In [1]:
%load_ext autoreload
%autoreload 2

import os, sys, random, logging

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0,1,2,6")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.append("../..")


In [2]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from rich.console import Console
from rich.theme import Theme
from rich.markup import escape

from src.utils import logging_utils

console = Console(theme=Theme({"default": "white"}), highlight=False)
def rich_print(*a, **k): console.print(*a, **k)

logging_utils.configure(level=logging.INFO)
logger = logging.getLogger(__name__)


In [3]:
from src.models import ModelandTokenizer

MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"
mt = ModelandTokenizer(
    model_key=MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
print(f"n_layer={mt.n_layer}, n_embd={mt.n_embd}, n_heads={mt.model.config.num_attention_heads}")


2026-07-05 11:20:28 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models
2026-07-05 11:20:28 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-05 11:20:28 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/config.json "HTTP/1.1 200 OK"
2026-07-05 11:20:28 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-05 11:20:28 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/tokenizer_config.json "HTTP/1.1 200 OK"
2026-07-05 11:20:28 ht

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

2026-07-05 11:20:40 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-05 11:20:40 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/generation_config.json "HTTP/1.1 200 OK"
2026-07-05 11:20:40 src.models INFO     loaded model <Qwen/Qwen2.5-Coder-32B-Instruct> | size: 62492.135 MB | dtype: torch.bfloat16 | device: cuda:0
n_layer=64, n_embd=5120, n_heads=40


## 1. Load the trained completion mask (Python-only) + selected heads

In [4]:
# ── loaders for completion-mask results (scripts/clock/train_completion_mask.py) ──
from src.goals.oid_gaze import cache_root
from src.goals.head_loc_attn import HeadMask

COMPLETION_VERSION = "conditions"
# swap to "completion_mask_gaze" if you want the gaze-obj variant instead
MASK_KIND = "completion_mask_gaze"
COMPLETION_MASK_ROOT = os.path.join(cache_root(MODEL_NAME, COMPLETION_VERSION), MASK_KIND)


def list_completion_masks():
    if not os.path.isdir(COMPLETION_MASK_ROOT):
        return []
    items = [
        (t, os.path.join(COMPLETION_MASK_ROOT, t),
         os.path.getmtime(os.path.join(COMPLETION_MASK_ROOT, t)))
        for t in os.listdir(COMPLETION_MASK_ROOT)
        if os.path.isdir(os.path.join(COMPLETION_MASK_ROOT, t))
    ]
    return sorted(items, key=lambda x: x[2], reverse=True)


def load_completion_mask(tag=None):
    items = list_completion_masks()
    assert items, f"no {MASK_KIND} results under {COMPLETION_MASK_ROOT}"
    if tag is None:
        d = items[0][1]
    else:
        matches = [it for it in items if tag in it[0]]
        assert matches, f"no run under {COMPLETION_MASK_ROOT} contains tag {tag!r}"
        d = matches[0][1]
    ck = torch.load(os.path.join(d, "mask.pt"), map_location="cpu", weights_only=False)
    ck["dir"] = d
    return ck


def completion_mask_to_headmask(ck):
    layers = [int(x) for x in ck["layers"]]
    nH = ck["mask_values"].shape[1]
    hm = HeadMask(len(layers), nH)
    with torch.no_grad():
        hm.values.copy_(ck["mask_values"])
    return hm


In [5]:
# ── pick a mask and inspect it ──
# COMPLETION_MASK_TAG = None    # newest by mtime
COMPLETION_MASK_TAG = "lam50.0"   # substring match on the run tag

CK     = load_completion_mask(COMPLETION_MASK_TAG)
layers = [int(x) for x in CK["layers"]]
mv     = CK["mask_values"].numpy()
hist   = CK["history"]
meta   = CK["meta"]
sel_heads    = [tuple(int(z) for z in lh) for lh in CK["decision_heads"]]
thr    = float(meta.get("sel_threshold", 0.9))

print(f"[{os.path.basename(CK['dir'])}]")
print(f"  layers {layers[0]}-{layers[-1]} ({len(layers)} masked) · patch_tokens={CK['patch_tokens']} · "
      f"direction={meta['direction']} · same={meta['same']} · N={meta['n_pairs']}")
print(f"  {len(sel_heads)} decision heads (mask>{thr}) · "
      f"final train score={hist['train_score'][-1]:+.4f} · val score={hist['val_score'][-1]:+.4f}")
ranked = sorted(sel_heads, key=lambda lh: mv[layers.index(lh[0]), lh[1]], reverse=True)
print("  top decision heads:", [f"L{l}H{h}" for (l, h) in ranked[:15]])


[diff_pt-2_both_L0-64_gaze100_lam50.0_n600]
  layers 0-63 (64 masked) · patch_tokens=[-2] · direction=both · same=False · N=600
  218 decision heads (mask>0.5) · final train score=+11.1219 · val score=+10.3482
  top decision heads: ['L3H27', 'L8H3', 'L13H22', 'L15H23', 'L20H28', 'L23H14', 'L24H1', 'L25H19', 'L27H16', 'L28H13', 'L28H29', 'L30H13', 'L30H29', 'L30H33', 'L31H10']


In [6]:
# ── selected heads (mask > thr) as a set + a binary HeadMask for patching ──
SEL_THRESHOLD = 0.8
n_layer_total = mt.model.config.num_hidden_layers
n_heads_total = mv.shape[1]

full_mask = np.full((n_layer_total, n_heads_total), np.nan, dtype=float)
for li, L in enumerate(layers):
    full_mask[L] = mv[li]

sel_set   = {(int(l), int(h)) for l, h in np.argwhere(full_mask > SEL_THRESHOLD)}
sel_heads = sorted(sel_set)

# Default set used for interventions in §3–§5. The separator-analysis cells below overwrite
# this with filtered_sel_heads; seeding it here keeps §3–§5 runnable even if that step is skipped.
DECISION_HEADS = sel_heads

mask_bin = HeadMask(len(layers), n_heads_total)
with torch.no_grad():
    mask_bin.values.zero_()
    for (l, h) in sel_set:
        mask_bin.values[layers.index(l), h] = 1.0
print(f"mask_bin: {int(mask_bin.values.sum().item())} heads = 1.0  (binary @ thr {SEL_THRESHOLD})")
print(f"DECISION_HEADS: {len(DECISION_HEADS)} heads (default = sel_heads; override via the filter cell)")


def get_mask_bin(heads):
    """HeadMask (on CPU) with 1.0 for `heads`, 0.0 elsewhere. Layer index space is `layers`."""
    hm = HeadMask(len(layers), n_heads_total)
    with torch.no_grad():
        hm.values.zero_()
        for (l, h) in heads:
            hm.values[layers.index(l), h] = 1.0
    return hm


mask_bin: 147 heads = 1.0  (binary @ thr 0.8)
DECISION_HEADS: 147 heads (default = sel_heads; override via the filter cell)


In [7]:
# -- selected heads: does OV write at the pivot separate stay (ml) vs move-on (sl)? --
from torch.nn.functional import cosine_similarity as _cos
from src.goals.completion_pairs import build_completion_pairs
from src.goals.head_loc_attn import collect_head_steering_vecs
from src.tokens import prepare_input

N_DECISION = 100
PIVOT_POS = -2

# minimal pairs: source IS dest (same spec, ml vs sl differ only at the target line)
dec_pairs = build_completion_pairs(
    mt, n_pairs=N_DECISION, direction="both", same=True,
    patch_tokens=tuple(int(t) for t in CK["patch_tokens"]), seed=2024,
)
stay_prompts = [p.source.ml_prompt for p in dec_pairs]    # incomplete target  -> stay/continue
move_prompts = [p.source.sl_prompt for p in dec_pairs]    # complete target    -> moved_on
print(f"{len(dec_pairs)} minimal pairs; reading {len(sel_heads)} selected heads at pos {PIVOT_POS}")

# per-head OV write at the pivot for both situations (left-pad => -2 aligns across lengths)
stay_tok = prepare_input(stay_prompts, tokenizer=mt, padding_side="left")
move_tok = prepare_input(move_prompts, tokenizer=mt, padding_side="left")
_, stay_h = collect_head_steering_vecs(mt, stay_tok, sel_heads, [PIVOT_POS], return_individual_heads=True)
_, move_h = collect_head_steering_vecs(mt, move_tok, sel_heads, [PIVOT_POS], return_individual_heads=True)


rows = []
for (l, h) in sorted(sel_heads):
    stay = stay_h[(l, h)][:, 0, :].float()   # (N, n_embd)
    move = move_h[(l, h)][:, 0, :].float()   # (N, n_embd)
    per_pair = _cos(stay, move, dim=1)                       # (N,)  cos within each spec
    mean_vs_mean = _cos(stay.mean(0), move.mean(0), dim=0).item()
    rows.append((l, h, per_pair.mean().item(), per_pair.std().item(), mean_vs_mean))

rows.sort(key=lambda r: r[2])   # most-separating (lowest cosine) first
print(f"\n{'layer':>5} {'head':>5} {'per-pair cos (mean+/-std)':>26} {'mean-vs-mean cos':>18}")
for l, h, m, s, mm in rows:
    print(f"{l:5d} {h:5d} {m:>14.3f} +/-{s:<7.3f} {mm:>18.3f}")

  built 25/100 pairs (27 attempts)
  built 50/100 pairs (54 attempts)
  built 75/100 pairs (79 attempts)
  built 100/100 pairs (104 attempts)
built 100 pairs from 104 attempts
100 minimal pairs; reading 147 selected heads at pos -2

layer  head  per-pair cos (mean+/-std)   mean-vs-mean cos
   44    28          0.120 +/-0.193                0.101
   31    10          0.129 +/-0.205                0.010
   13    22          0.174 +/-0.214                0.221
   40    32          0.177 +/-0.222                0.084
   48    26          0.178 +/-0.197                0.266
   14    25          0.179 +/-0.204                0.327
   48    28          0.181 +/-0.210                0.210
    1    28          0.203 +/-0.496                0.332
   51     2          0.207 +/-0.299                0.307
   28    13          0.228 +/-0.221                0.223
   43    16          0.246 +/-0.196                0.279
   38    29          0.247 +/-0.218                0.072
   34    17          0.26

In [8]:
filtered_sel_heads = [
    (l, h)
    for (l, h), m, s, mm in zip(
        sel_heads, [r[2] for r in rows], [r[3] for r in rows], [r[4] for r in rows]
    )
    if m < 0.7
]
print(len(filtered_sel_heads), len(sel_heads))


#######################################################################
# DECISION_HEADS = filtered_sel_heads
DECISION_HEADS = sel_heads
#######################################################################
len(DECISION_HEADS)

113 147


147

In [9]:
from src.goals.head_loc_attn import HeadMask
from src.utils import env_utils
import matplotlib.patches as mpatches
from scripts.clock.score_condition_heads import load_head_scores


# ── pick the scoring config to inspect ────────────────────────────────────────
Q_SPAN = 15
COMPARE_WITH = "mean"        # "mean" | "max"
TOP_K = 100
VERSION = "conditions"
# ──────────────────────────────────────────────────────────────────────────────
gaze_scores = load_head_scores(model=MODEL_NAME, version=VERSION, q_span=Q_SPAN, compare_with=COMPARE_WITH)["combined"]

# ── top-K gaze heads ──────────────────────────────────────────────────────────
flat_idx = np.argsort(gaze_scores, axis=None)[::-1][:TOP_K]
gaze_heads = [(int(l), int(h)) for l, h in (np.unravel_index(i, gaze_scores.shape) for i in flat_idx)]
print(f"top-{TOP_K} gaze heads  (compare_with='{COMPARE_WITH}', q_span={Q_SPAN})  "
      f"score range [{gaze_scores[gaze_heads[-1]]:.3f}, {gaze_scores[gaze_heads[0]]:.3f}]")
for rank, (l, h) in enumerate(gaze_heads[:15], start=1):
    print(f"  {rank:2d}. L{l:2d} H{h:2d}  score={gaze_scores[l, h]:.3f}")
if TOP_K > 15:
    print(f"  ... (+ {TOP_K - 15} more)")

top-100 gaze heads  (compare_with='mean', q_span=15)  score range [2.725, 3.478]
   1. L53 H16  score=3.478
   2. L52 H23  score=3.464
   3. L40 H21  score=3.452
   4. L51 H13  score=3.451
   5. L36 H23  score=3.422
   6. L59 H34  score=3.364
   7. L34 H11  score=3.358
   8. L45 H12  score=3.293
   9. L48 H 8  score=3.277
  10. L51 H11  score=3.246
  11. L45 H11  score=3.222
  12. L48 H 5  score=3.217
  13. L50 H12  score=3.181
  14. L32 H23  score=3.175
  15. L53 H18  score=3.168
  ... (+ 85 more)


## 2. Per-head **mean** intervention signal — computed on Python condition-check

Same recipe as `04.3_eval` §6.1: pull the source o_proj-input at the pivot for `N_MEAN_SRC`
sources drawn from `build_completion_pairs`, average over sources → one `(n_layers, n_embd)` tensor
that acts as the "general" source signal.  Two choices:

- `HP_DIRECTION = "sl->ml"`  → source style = `sl` (moved_on)  → mean = **move-on** signal → pushes a dest to move on
- `HP_DIRECTION = "ml->sl"`  → source style = `ml` (multiline/stay) → mean = **stay** signal → pushes a dest to stay

Then in §3–§5 we patch selected heads' OV at the pivot with this mean signal on OOD dests.

In [10]:
from src.goals.completion_pairs import build_completion_pairs
from src.goals.head_loc_attn import get_o_proj_inputs_batch
from src.tokens import prepare_input

# ── config for the mean signal ──
HP_DIRECTION   = "sl->ml"          # "sl->ml" = MOVE-ON mean (push a stay dest to move on)
                                   # "ml->sl" = STAY    mean (push a move-on dest to stay)
PATCH_POS      = int(CK["patch_tokens"][0]) if len(CK["patch_tokens"]) == 1 else -2
N_MEAN_SRC     = 200
MEAN_SRC_BATCH = 16
MEAN_SEED      = 1234

src_style = "sl" if HP_DIRECTION == "sl->ml" else "ml"
sig_name  = "move-on" if src_style == "sl" else "stay"
print(f"target: mean '{sig_name}' signal ({HP_DIRECTION}) at POS={PATCH_POS} over N={N_MEAN_SRC}")

target: mean 'move-on' signal (sl->ml) at POS=-2 over N=200


In [11]:
# ── compute the mean signal, or load from local disk cache if we already did ──
import json, hashlib
from src.utils import env_utils

OVERWRITE_MEAN = False   # set True to force recompute

NB_CACHE_DIR = os.path.join(env_utils.DEFAULT_RESULTS_DIR, "notebook_cache")
os.makedirs(NB_CACHE_DIR, exist_ok=True)

_mean_cfg = dict(
    model=MODEL_NAME, kind="mean_signal", direction=HP_DIRECTION, n_src=N_MEAN_SRC,
    same=bool(meta.get("same", False)),
    patch_tokens=tuple(int(t) for t in CK["patch_tokens"]),
    patch_pos=int(PATCH_POS), layers=list(int(l) for l in layers), seed=MEAN_SEED,
)
_key = hashlib.md5(json.dumps(_mean_cfg, sort_keys=True, default=str).encode()).hexdigest()[:10]
_mean_path = os.path.join(
    NB_CACHE_DIR,
    f"mean_signal_{HP_DIRECTION.replace('->', '_')}_N{N_MEAN_SRC}_{_key}.pt",
)

if not OVERWRITE_MEAN and os.path.exists(_mean_path):
    mean_signal = torch.load(_mean_path, map_location="cpu")
    print(f"loaded mean_signal from cache: {_mean_path}")
    print("mean_signal:", tuple(mean_signal.shape))
else:
    mean_pairs = build_completion_pairs(
        mt, n_pairs=N_MEAN_SRC, direction=HP_DIRECTION,
        same=bool(meta.get("same", False)),
        patch_tokens=tuple(int(t) for t in CK["patch_tokens"]),
        seed=MEAN_SEED,
    )
    mean_src_prompts = [p.source.prompt(src_style) for p in mean_pairs]
    print(f"sampled {len(mean_src_prompts)} sources (style={src_style}) for the mean '{sig_name}' signal")

    # batched o_proj cache at PATCH_POS across all masked layers -> mean over sources
    # get_o_proj_inputs_batch(..., token_positions=[POS]) returns {layer: (b, 1, n_embd)};
    # stack over layers and index the singleton position dim -> (b, n_layers, n_embd).
    _acc, n_seen = None, 0
    for i in range(0, len(mean_src_prompts), MEAN_SRC_BATCH):
        chunk = mean_src_prompts[i : i + MEAN_SRC_BATCH]
        c = get_o_proj_inputs_batch(mt, chunk, layers, token_positions=[PATCH_POS])
        stk = torch.stack([c[l][:, 0] for l in layers], dim=1)  # (b, n_layers, n_embd)
        _acc = stk.sum(0) if _acc is None else _acc + stk.sum(0)
        n_seen += stk.shape[0]
    mean_signal = (_acc / n_seen).float()  # (n_layers, n_embd)
    torch.save(mean_signal, _mean_path)
    print("mean_signal shape:", tuple(mean_signal.shape), "from", n_seen, "sources")
    print(f"saved mean_signal to cache: {_mean_path}")

loaded mean_signal from cache: /disk/u/arnab/Codes/Projects/goals/results/notebook_cache/mean_signal_sl_ml_N200_bb320d66a8.pt
mean_signal: (64, 5120)


## 3. In-distribution sanity — python `build_example`

If everything is wired up correctly, this should look the same as `04.3_eval` §6.1 (mean-replace):
patching the selected heads' OV with the move-on mean at the pivot should visibly redirect the
dest away from the "stay/continue" continuation.

In [12]:
def test_aug(prompt: str) -> str:
    """Turn each `\\n` inside the final <code> block into `    \\n` so trailing-whitespace
    tokenization is stable (used before tokenizing for latent extraction / patching)."""
    code_split = prompt.split("<code>")
    code_segment = code_split[-1]
    prefix = "<code>".join(code_split[:-1])
    code_segment = code_segment.replace("\n", " #\n")
    return prefix + "<code>" + code_segment

In [13]:
from src.functional import generate_with_patch
from src.goals.multiline_prompts import build_example, replace_in_code_newline_with_tabnewline
from src.goals.goalscope import headpatched_generate

def render_prompt(example):
    """apply the augmentation + trailing `    ` so PIVOT=-2 lands on `    \n` at the pivot."""
    return replace_in_code_newline_with_tabnewline(example["full_prompt"]) + "   "


def run_intervention(tok, decision_heads, n_gen=32, patch_pos=[PATCH_POS], alpha=1.0):
    """Clean vs mean-replace generation for a single prompt."""
    clean = generate_with_patch(
        mt, inputs=tok, n_gen_per_prompt=1, max_new_tokens=n_gen,
        do_sample=False, remove_prefix=True,
    )[0]
    patched = headpatched_generate(
        mt, dest_inputs=tok, src_o_proj_inps=mean_signal,
        mask=get_mask_bin(decision_heads),
        layers=layers, max_new_tokens=n_gen,
        n_steer_tokens=1, patch_input_positions=patch_pos,
        alpha=alpha
    )
    return clean, patched


NAMING = "natural"
N_GEN  = 32

# dest style is the OPPOSITE of source (we're pushing dest to flip)
dst_style_full = "singleline" if src_style == "ml" else "multiline"

example_py = build_example(
    mt=mt, rng=random.Random(135),
    target_slot=1, n_conditions=4,
    mode="chat", style=dst_style_full, naming=NAMING, lang="python",
)
# prompt_py = render_prompt(example_py)
prompt_py = test_aug(example_py["full_prompt"]) + "   " 
tok_py = prepare_input(prompt_py, tokenizer=mt, padding_side="left")

clean_py, patched_py = run_intervention(tok_py, DECISION_HEADS, n_gen=N_GEN)
rich_print(f"[bold]python | clean          [/bold] [white]{escape(prompt_py)}[/white][sea_green3]{escape(clean_py)}[/sea_green3]")
rich_print(f"[bold]python | mean-replace ({len(DECISION_HEADS)})[/bold] [white]{escape(prompt_py)}[/white][salmon1]{escape(patched_py)}[/salmon1]")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


python | clean           <|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a Python program that implements a function `check_conditions` which, given an integer n, checks the 
following conditions:

1. n is coprime with 6
2. n has at least 3 digits
3. all of n's digits are even
4. n has more than 3 ones in its binary representation

For each condition, in the order listed, assign the boolean result of the check to its own local variable. After 
all conditions have been checked, use a SINGLE print statement that, using those variables, reports whether each 
condition passed individually and whether all of them passed collectively.
Compute each condition's boolean result inside `check_conditions` (one local variable per condition); do not define
any helper functions.
Do not include any comments in your code.<|im_end|>
<|im_start|>assistant
<code> #
def check_conditions(n: int): #
    is_coprime = math.gcd(n, 6) #
    is_coprime = is_coprime == 1 #
    has_min_digits = len(str(abs(n))) #
    has_min_digits = has_min_digits >= 3 #
    all_even_digits = all(int(digit) % 2 == 0 for digit in str(abs

python | mean-replace (147) <|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a Python program that implements a function `check_conditions` which, given an integer n, checks the 
following conditions:

1. n is coprime with 6
2. n has at least 3 digits
3. all of n's digits are even
4. n has more than 3 ones in its binary representation

For each condition, in the order listed, assign the boolean result of the check to its own local variable. After 
all conditions have been checked, use a SINGLE print statement that, using those variables, reports whether each 
condition passed individually and whether all of them passed collectively.
Compute each condition's boolean result inside `check_conditions` (one local variable per condition); do not define
any helper functions.
Do not include any comments in your code.<|im_end|>
<|im_start|>assistant
<code> #
def check_conditions(n: int): #
    is_coprime = math.gcd(n, 6) #
    is_coprime = is_coprime == 1 #
    has_min_digits = len(str(abs(n))) #
    all_even_digits = all(int(digit) % 2 == 0 for digit in str(abs(n))) #
    binary_ones_count = bin(n).

In [ ]:
# ── §3 attention: clean vs mean-replace on the python dest ────────────────────
from src.attention import get_attention_matrices, show_interactive_attention
from src.goals.head_loc_attn import make_head_replacement_patch_interface

P_py = len(tok_py.input_ids[0])                      # dest-prompt length
pivot_abs_py = P_py + PATCH_POS                      # absolute index of the patched pivot

# guard: prompt must tokenize identically as a prefix of prompt+gen so pivot_abs is correct
_clean_ids = prepare_input(prompt_py + clean_py, tokenizer=mt).input_ids[0]
assert _clean_ids[:P_py].tolist() == tok_py.input_ids[0].tolist(), \
    "retokenization drift: prompt_py is not a clean prefix of prompt_py + clean_py"

clean_attn_py = get_attention_matrices(
    input=prompt_py + clean_py, mt=mt, value_weighted=True,
)
show_interactive_attention(
    attn_info=clean_attn_py,
    selected_heads=sorted(DECISION_HEADS),
    start_from=0, agg="mean", normalize="row_max", cmap="Greens",
    title=f"python | clean attention — {len(DECISION_HEADS)} decision heads",
    track_conditions=example_py["spec"],
    intervened_positions=[pivot_abs_py],
    height=None,
)

# head-patched attention via the SAME head-replacement transplant used for steering
_patched_ids = prepare_input(prompt_py + patched_py, tokenizer=mt).input_ids[0]
assert _patched_ids[:P_py].tolist() == tok_py.input_ids[0].tolist(), \
    "retokenization drift: prompt_py is not a clean prefix of prompt_py + patched_py"

patch_iface_py = make_head_replacement_patch_interface(
    src_head_inps=mean_signal,
    mask=get_mask_bin(DECISION_HEADS),
    layers=layers,
    patch_positions=[pivot_abs_py],
)
patched_attn_py = get_attention_matrices(
    input=prompt_py + patched_py, mt=mt, value_weighted=True,
    patch_interface=patch_iface_py,
)
show_interactive_attention(
    attn_info=patched_attn_py,
    selected_heads=sorted(DECISION_HEADS),
    start_from=0, agg="mean", normalize="row_max", cmap="Reds",
    title=f"python | mean-replace attention — {len(DECISION_HEADS)} decision heads",
    track_conditions=example_py["spec"],
    intervened_positions=[pivot_abs_py],
    height=None,
)


## 4. Same shape, new language — cpp `build_example`

`condition_data.py` and `multiline_prompts.py` now support `lang="cpp"`.  The dest prompt is a
`check_conditions(int n)` C++ program with the same conditions as §3, formatted as
`multiline` vs `singleline` in exactly the same way.  The intervention direction and heads are
**identical** to §3 — no retraining, no re-cache.  The question is whether the same
`mean_signal` (learned on python) still redirects the dest.


In [ ]:
# ── build a cpp dest example and run the same intervention as §3 ─────────────
example_cpp = build_example(
    mt=mt, rng=random.Random(135),
    target_slot=1, n_conditions=4,
    mode="chat", style=dst_style_full, naming=NAMING, lang="cpp",
)
prompt_cpp = test_aug(example_cpp["full_prompt"]) + "   "
tok_cpp = prepare_input(prompt_cpp, tokenizer=mt, padding_side="left")

clean_cpp, patched_cpp = run_intervention(tok_cpp, DECISION_HEADS, n_gen=N_GEN)
rich_print(f"[bold]cpp | clean          [/bold] [white]{escape(prompt_cpp)}[/white][sea_green3]{escape(clean_cpp)}[/sea_green3]")
rich_print(f"[bold]cpp | mean-replace ({len(DECISION_HEADS)})[/bold] [white]{escape(prompt_cpp)}[/white][salmon1]{escape(patched_cpp)}[/salmon1]")


In [ ]:
# ── §4 attention: clean vs mean-replace on the cpp dest ──────────────────────
P_cpp = len(tok_cpp.input_ids[0])
pivot_abs_cpp = P_cpp + PATCH_POS

_clean_ids = prepare_input(prompt_cpp + clean_cpp, tokenizer=mt).input_ids[0]
assert _clean_ids[:P_cpp].tolist() == tok_cpp.input_ids[0].tolist(), \
    "retokenization drift: prompt_cpp is not a clean prefix of prompt_cpp + clean_cpp"

clean_attn_cpp = get_attention_matrices(
    input=prompt_cpp + clean_cpp, mt=mt, value_weighted=True,
)
show_interactive_attention(
    attn_info=clean_attn_cpp,
    selected_heads=sorted(DECISION_HEADS),
    start_from=0, agg="mean", normalize="row_max", cmap="Greens",
    title=f"cpp | clean attention — {len(DECISION_HEADS)} decision heads",
    track_conditions=example_cpp["spec"],
    intervened_positions=[pivot_abs_cpp],
    height=None,
)

_patched_ids = prepare_input(prompt_cpp + patched_cpp, tokenizer=mt).input_ids[0]
assert _patched_ids[:P_cpp].tolist() == tok_cpp.input_ids[0].tolist(), \
    "retokenization drift: prompt_cpp is not a clean prefix of prompt_cpp + patched_cpp"

patch_iface_cpp = make_head_replacement_patch_interface(
    src_head_inps=mean_signal,
    mask=get_mask_bin(DECISION_HEADS),
    layers=layers,
    patch_positions=[pivot_abs_cpp],
)
patched_attn_cpp = get_attention_matrices(
    input=prompt_cpp + patched_cpp, mt=mt, value_weighted=True,
    patch_interface=patch_iface_cpp,
)
show_interactive_attention(
    attn_info=patched_attn_cpp,
    selected_heads=sorted(DECISION_HEADS),
    start_from=0, agg="mean", normalize="row_max", cmap="Reds",
    title=f"cpp | mean-replace attention — {len(DECISION_HEADS)} decision heads",
    track_conditions=example_cpp["spec"],
    intervened_positions=[pivot_abs_cpp],
    height=None,
)


## 5. Hardcoded C++ prompts

Freeform C++ programs — not the `check_conditions(int n)` shape.  This tests whether the
intervention still fires when the surrounding scaffolding, variable names, and control flow
diverge from what the mask was trained on.

**Fill in your own C++ examples below.**  The scaffold applies the same augmentation +
`PATCH_POS=-2` pivot as §3/§4.  For each prompt you supply:
- `<code>` tag wrapping the C++ block (matches `apply_chat_template`)
- the last line before the pivot should be an incomplete statement where a "stay" (continue the
  current line) vs "move-on" (start a new statement) decision is meaningful
- ensure the last two chars are `    ` (four spaces) so PIVOT=-2 lands on the pivot whitespace

Without an aligned pivot the intervention position is off and the whole comparison is invalid.


In [ ]:
# ── hardcoded C++ prompts — REPLACE THESE STRINGS with your own examples ─────
# each string must:
#   (a) already be a full chat-formatted prompt (system + user + assistant tags), OR
#   (b) be a bare C++ code block, in which case wrap it via apply_chat_template below
from src.goals.functional import apply_chat_template

CPP_INCOMPLETE_PROGRAMS = [
    # TODO: paste your actual C++ prompts here.  Example placeholder:
    r"""#include <iostream>
int main() {
    int n = 42;
    bool a = (n % 2 == 0);
    bool b =
""",
    # TODO: add more.  Each entry gets its own intervention + attn-viz pass below.
]

# turn each bare code block into a chat-formatted prompt (matches build_example)
CPP_PROMPTS = []
for code in CPP_INCOMPLETE_PROGRAMS:
    full = apply_chat_template(
        mt=mt,
        user_query="Complete this C++ program.",
        incomplete_code=code,
    )
    # same test_aug + trailing spaces pattern as §3/§4 so PIVOT=-2 stays valid
    CPP_PROMPTS.append(test_aug(full) + "   ")

print(f"prepared {len(CPP_PROMPTS)} hardcoded cpp prompts")


In [ ]:
# ── clean vs mean-replace on each hardcoded C++ prompt ───────────────────────
cpp_results = []
for i, p in enumerate(CPP_PROMPTS):
    tok = prepare_input(p, tokenizer=mt, padding_side="left")
    clean, patched = run_intervention(tok, DECISION_HEADS, n_gen=N_GEN)
    cpp_results.append({"prompt": p, "tok": tok, "clean": clean, "patched": patched})
    rich_print(f"\n[bold]cpp[{i}] | clean          [/bold] [white]{escape(p)}[/white][sea_green3]{escape(clean)}[/sea_green3]")
    rich_print(f"[bold]cpp[{i}] | mean-replace ({len(DECISION_HEADS)})[/bold] [white]{escape(p)}[/white][salmon1]{escape(patched)}[/salmon1]")


In [ ]:
# ── attention viz for each hardcoded C++ prompt ──────────────────────────────
# no `track_conditions` here — the hardcoded prompts don't come with a ConditionSpec.
for i, r in enumerate(cpp_results):
    P_i = len(r["tok"].input_ids[0])
    pivot_abs_i = P_i + PATCH_POS

    _cids = prepare_input(r["prompt"] + r["clean"], tokenizer=mt).input_ids[0]
    assert _cids[:P_i].tolist() == r["tok"].input_ids[0].tolist(), \
        f"retokenization drift on cpp[{i}] (clean)"
    clean_attn = get_attention_matrices(
        input=r["prompt"] + r["clean"], mt=mt, value_weighted=True,
    )
    show_interactive_attention(
        attn_info=clean_attn,
        selected_heads=sorted(DECISION_HEADS),
        start_from=0, agg="mean", normalize="row_max", cmap="Greens",
        title=f"cpp[{i}] | clean attention — {len(DECISION_HEADS)} decision heads",
        intervened_positions=[pivot_abs_i],
        height=None,
    )

    _pids = prepare_input(r["prompt"] + r["patched"], tokenizer=mt).input_ids[0]
    assert _pids[:P_i].tolist() == r["tok"].input_ids[0].tolist(), \
        f"retokenization drift on cpp[{i}] (patched)"
    patch_iface_i = make_head_replacement_patch_interface(
        src_head_inps=mean_signal,
        mask=get_mask_bin(DECISION_HEADS),
        layers=layers,
        patch_positions=[pivot_abs_i],
    )
    patched_attn = get_attention_matrices(
        input=r["prompt"] + r["patched"], mt=mt, value_weighted=True,
        patch_interface=patch_iface_i,
    )
    show_interactive_attention(
        attn_info=patched_attn,
        selected_heads=sorted(DECISION_HEADS),
        start_from=0, agg="mean", normalize="row_max", cmap="Reds",
        title=f"cpp[{i}] | mean-replace attention — {len(DECISION_HEADS)} decision heads",
        intervened_positions=[pivot_abs_i],
        height=None,
    )
